In [ ]:
pip install natsort

In [21]:
import tensorflow as tf

def gen_loss(D_fake):
    # Generator loss: maximize the fake points' score from the critic (G tries to fool D)
    return -tf.reduce_mean(D_fake)

def critic_loss_gp(D_real, D_fake, Y, Y_cap, model, batch_size):
    # Critic loss with gradient penalty
    dloss = tf.reduce_mean(D_fake) - tf.reduce_mean(D_real)
    lam = 10  # Lambda for gradient penalty
    eps = tf.random.uniform(shape=[batch_size, 1, 1], minval=0, maxval=1)
    x_cap = eps * Y + (1 - eps) * Y_cap  # Interpolation between real and fake

    with tf.GradientTape() as gptape:
        gptape.watch(x_cap)
        out = model.critic(x_cap, training=True)  # Compute critic output for interpolated points
    grad = gptape.gradient(out, x_cap)[0]
    grad_norm = tf.sqrt(tf.reduce_sum(tf.square(grad), axis=[0, 1]))
    grad_pen = tf.reduce_mean((grad_norm - 1.0) ** 2)  # Gradient penalty
    dloss = dloss + lam * grad_pen  # Add penalty to the loss
    return dloss


In [38]:
from glob import glob
from natsort import natsorted
import os
import numpy as np
import tensorflow as tf
import functools
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

batch_size = 16

def fetch_data(path, cat_dict):
    path = path.numpy().decode('utf-8')
    X = np.load(path)  # Load the 3D point cloud data
    # print(cat_dict, 'hiiiiiiii')
    Y = cat_dict[path.split(os.path.sep)[-3]]  # Get category
    return X, Y

def load_batch(path, batch_size=16):
    return tf.data.Dataset.from_tensor_slices((path)).shuffle(len(path)).batch(batch_size, drop_remainder=True)

def load_data(path, cat_dict):
    X, Y = list(zip(*map(functools.partial(fetch_data, cat_dict=cat_dict), path)))
    X = tf.convert_to_tensor(X, dtype='float32')
    Y = tf.convert_to_tensor(Y, dtype='int32')
    return X, Y



def view_data(X, Y, cat_dict, rev_cat_dict):
    if not os.path.isdir('save_fig'):
        os.makedirs('save_fig')

    for idx, (shape, cat) in enumerate(zip(X, Y)):
        shape = shape.numpy()
        cat = cat.numpy()
        ax = plt.axes(projection='3d')
        ax.scatter3D(shape[:, 0], shape[:, 1], shape[:, 2])  # Plot 3D points
        plt.title(f'Category: {rev_cat_dict[cat]}')

        # Adjust axis ticks to prevent label overlap
        ax.xaxis.set_major_locator(MaxNLocator(nbins=5))  # Limit number of x-axis ticks
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))  # Limit number of y-axis ticks
        ax.zaxis.set_major_locator(MaxNLocator(nbins=5))  # Limit number of z-axis ticks

        plt.savefig(f'save_fig/{idx}.png')
        plt.close()  # Close the figure to prevent overlap in subsequent plots

def view_results(model, latent_dim, batch_size, categ, fileno):
    if not os.path.isdir('results'):
        os.makedirs('results/train/')
        os.makedirs('results/val/')

    idx = np.random.randint(0, batch_size - 1)
    latent_input = tf.random.normal([batch_size, latent_dim])  # Latent input for generation
    generated_points = model.generator(latent_input, training=False)
    X = generated_points[idx].numpy()  # 3D points

    ax = plt.axes(projection='3d')
    ax.scatter3D(X[:, 0], X[:, 1], X[:, 2])  # Scatter plot 3D points

    # Adjust axis ticks to prevent label overlap
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))  # Limit number of x-axis ticks
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))  # Limit number of y-axis ticks
    ax.zaxis.set_major_locator(MaxNLocator(nbins=5))  # Limit number of z-axis ticks

    plt.savefig(f'results/{categ}/{fileno}.png')
    plt.close()  # Close the figure to prevent memory issues


In [40]:
import tensorflow as tf
from tensorflow.keras import layers

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class TransformerGenerator(tf.keras.Model):
    def __init__(self, embed_dim, num_heads, ff_dim, num_layers, latent_dim, output_dim, num_points=2048):
        super(TransformerGenerator, self).__init__()
        self.latent_layer = layers.Dense(embed_dim)
        self.expand = layers.Dense(num_points * embed_dim)  # Expand to sequence length
        self.reshape = layers.Reshape((num_points, embed_dim))
        self.transformer_blocks = [TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)]
        self.output_layer = layers.Dense(output_dim)
        self.num_points = num_points

    def call(self, latent_input, training=True):
        # Ensure that the layers are only created once during model initialization
        x = self.latent_layer(latent_input)  # latend_input is passed from the train_step
        x = self.expand(x)
        x = self.reshape(x)
        for transformer_block in self.transformer_blocks:
            x = transformer_block(x, training=training)
        generated_points = self.output_layer(x)
        return tf.reshape(generated_points, [-1, self.num_points, 3])


class TransformerCritic(tf.keras.Model):
    def __init__(self, embed_dim, num_heads, ff_dim, num_layers, num_points=2048):
        super(TransformerCritic, self).__init__()
        self.point_cloud_projection = layers.Dense(embed_dim)
        self.transformer_blocks = [TransformerBlock(embed_dim, num_heads, ff_dim) for _ in range(num_layers)]
        self.flatten = layers.Flatten()
        self.output_layer = layers.Dense(1)
        self.num_points = num_points

    def call(self, point_cloud, training=True):
        x = self.point_cloud_projection(point_cloud)
        for transformer_block in self.transformer_blocks:
            x = transformer_block(x, training=training)
        x = self.flatten(x)
        return self.output_layer(x)

class TransGAN(tf.keras.Model):
    def __init__(self, latent_dim, embed_dim=128, num_heads=4, ff_dim=256, num_layers=4, output_dim=3):
        super(TransGAN, self).__init__()
        self.latent_dim = latent_dim
        self.generator = TransformerGenerator(embed_dim, num_heads, ff_dim, num_layers, latent_dim, output_dim, num_points=2048)
        self.critic = TransformerCritic(embed_dim, num_heads, ff_dim, num_layers, num_points=2048)

    def generate(self, batch_size):
        latent_input = tf.random.normal([batch_size, self.latent_dim])
        return self.generator(latent_input, training=True)

    def call(self, inputs):
        generated_points = self.generator(inputs)
        critic_output = self.critic(generated_points)
        return critic_output

# Instantiate the model
latent_dim = 96
embed_dim = 128
num_heads = 4
ff_dim = 256
num_layers = 4
output_dim = 3  # 3D point coordinates (x, y, z)

model = TransGAN(latent_dim, embed_dim, num_heads, ff_dim, num_layers, output_dim)

# Optimizers
gen_opt = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)
critic_opt = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)

# Define the training and validation steps (train_step and val_step)
@tf.function
def train_step(model, gen_opt, critic_opt, real_points, latent_dim, batch_size):
    real_points = tf.ensure_shape(real_points, [batch_size, 2048, 3])  # Adjusted to 2048 points
    latent_input = tf.random.normal([batch_size, latent_dim])  # Latent input for generation

    with tf.GradientTape() as gen_tape, tf.GradientTape() as critic_tape:
        generated_points = model.generator(latent_input, training=True)  # Now the generator uses latent_input
        generated_points = tf.ensure_shape(generated_points, [batch_size, 2048, 3])  # Adjusted to 2048 points

        real_output = model.critic(real_points, training=True)
        fake_output = model.critic(generated_points, training=True)

        gen_loss = -tf.reduce_mean(fake_output)
        critic_loss = tf.reduce_mean(fake_output) - tf.reduce_mean(real_output)

    gen_gradients = gen_tape.gradient(gen_loss, model.generator.trainable_variables)
    critic_gradients = critic_tape.gradient(critic_loss, model.critic.trainable_variables)

    gen_opt.apply_gradients(zip(gen_gradients, model.generator.trainable_variables))
    critic_opt.apply_gradients(zip(critic_gradients, model.critic.trainable_variables))

    return gen_loss, critic_loss

@tf.function
def val_step(model, real_points, latent_dim, batch_size):
    latent_input = tf.random.normal([batch_size, latent_dim])
    generated_points = model.generator(latent_input, training=False)
    generated_points = tf.ensure_shape(generated_points, [batch_size, 2048, 3])  # Adjusted to 2048 points
    fake_output = model.critic(generated_points, training=False)
    return -tf.reduce_mean(fake_output)


In [ ]:
import tensorflow as tf
import os
from natsort import natsorted
from glob import glob
# from utils import load_data, load_batch, view_results
# from model import TransGAN, train_step, val_step
from tqdm import tqdm

os.environ["CUDA_DEVICE_ORDER"]= "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= '1'

if __name__ == '__main__':
    batch_size   = 16
    latent_dim   = 96
    train_path   = natsorted(glob('/kaggle/input/transgan2/Trans-GAN-main/data/*/*/train/*.npy'))
    val_path     = natsorted(glob('/kaggle/input/transgan2/Trans-GAN-main/data/*/*/test/*.npy'))
    cat_dict     = {}  # Category dictionary from class to idx
    rev_cat_dict = {}  # Category dictionary from idx to class

    for idx, cat in enumerate(natsorted(glob('/kaggle/input/transgan2/Trans-GAN-main/data/ModelNet10/*'))):
        # print(idx, cat)
        cat_dict[cat.split(os.path.sep)[-1]] = idx
        
        rev_cat_dict[idx] = cat.split(os.path.sep)[-1]

    # print(cat_dict)
    # Prepare train and validation batches
    train_batch = load_batch(train_path, batch_size=batch_size)
    val_batch   = load_batch(val_path, batch_size=batch_size)

    # Initialize TransGAN model
    model = TransGAN(latent_dim)

    # Optimizers
    gen_opt     = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)
    critic_opt  = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.5, beta_2=0.9)

    # Checkpoint and manager
    ckpt = tf.train.Checkpoint(step=tf.Variable(1), model=model, gopt=gen_opt, copt=critic_opt)
    manager = tf.train.CheckpointManager(ckpt, directory='transgan_ckpt', max_to_keep=10)
    ckpt.restore(manager.latest_checkpoint).expect_partial()

    EPOCHS = 100
    START = int(ckpt.step) // len(train_batch) + 1
    save_freq = 10
    tvis_freq = 10
    vvis_freq = 10

    if manager.latest_checkpoint:
        print('Restored from last checkpoint, epoch : {0}'.format(START))

    # Training loop
    for epoch in range(START, EPOCHS):
        print('epoch:', epoch)
        train_gloss = tf.keras.metrics.Mean()
        train_closs = tf.keras.metrics.Mean()
        val_gloss   = tf.keras.metrics.Mean()

        for idx, path in enumerate(tqdm(train_batch), start=1):
            X, Y = load_data(path, cat_dict)  # Load data for training
            gloss, closs = train_step(model, gen_opt, critic_opt, X, latent_dim, batch_size)
            train_gloss.update_state(gloss)
            train_closs.update_state(closs)
            ckpt.step.assign_add(1)

            if (idx % save_freq) == 0:
                manager.save()
            if (idx % tvis_freq) == 0:
                view_results(model, latent_dim, batch_size, 'train', int(ckpt.step))

            print('Train_GLoss: {0}\tTrain_CLoss: {1}'.format(gloss, closs))

        # Validation step
        for idx, path in enumerate(tqdm(val_batch), start=1):
            X, Y = load_data(path, cat_dict)  # Load data for validation
            gloss = val_step(model, X, latent_dim, batch_size)
            val_gloss.update_state(gloss)

            if (idx % vvis_freq) == 0:
                view_results(model, latent_dim, batch_size, 'val', int(ckpt.step) + idx)

            print('Val_GLoss: {0}'.format(gloss))

        # Logging the results
        with open('log.txt', 'a') as file:
            file.write('Epoch: {0}\tTrain_GLoss: {1}\tTrain_CLoss: {2}\tVal_GLoss: {3}\n'.format(
                epoch, train_gloss.result(), train_closs.result(), val_gloss.result()))

        print('Epoch: {0}\tTrain_GLoss: {1}\tTrain_CLoss: {2}\tVal_GLoss: {3}'.format(
            epoch, train_gloss.result(), train_closs.result(), val_gloss.result()))


Restored from last checkpoint, epoch : 1
epoch: 1


  3%|▎         | 1/31 [00:53<26:52, 53.75s/it]

Train_GLoss: 4.995064735412598	Train_CLoss: 3.4056034088134766


  6%|▋         | 2/31 [01:20<18:13, 37.71s/it]

Train_GLoss: -3.337510585784912	Train_CLoss: -15.075399398803711


...

Train_GLoss: 1706.388671875	Train_CLoss: -3425.142822265625


 13%|█▎        | 4/31 [01:46<11:59, 26.66s/it]

Train_GLoss: 1731.9755859375	Train_CLoss: -3475.57470703125


 16%|█▌        | 5/31 [02:12<11:30, 26.57s/it]

Train_GLoss: 1767.8785400390625	Train_CLoss: -3534.85498046875
